# RBF Manual

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 1. Carga y Preprocesamiento de Datos
Cargar el dataset desde el archivo JSON y extraer las entradas (X) y salidas deseadas (YD).

In [ ]:
with open('../jsons/dataset_rbf_1.json', 'r') as f:
    data_dict = json.load(f)

dataset_name = data_dict['dataset']
print(f"=== CARGA DE DATASET ===")
print(f"Dataset: {dataset_name}")
print(f"Features: {data_dict['features']}")
print(f"Número de muestras: {len(data_dict['data'])}")

X = []
YD = []
for item in data_dict['data']:
    X.append(item['input'])
    YD.append(item['output'])

X = np.array(X)
YD = np.array(YD).reshape(-1, 1)

print(f"\nDimensiones:")
print(f"  X (entradas): {X.shape}")
print(f"  YD (salidas deseadas): {YD.shape}")
print(f"\nNúmero de entradas: {X.shape[1]}")
print(f"Número de salidas: {YD.shape[1]}")
print(f"Número de patrones: {X.shape[0]}")
print(f"\nPrimeros 5 patrones:")
for i in range(min(5, len(X))):
    print(f"  Patrón {i+1}: X={X[i]}, YD={YD[i][0]}")

## 3. Estadística descriptiva

In [ ]:
# Estadística descriptiva
print(f"\nEstadísticas por característica:")
print(f"  {'Feature':<10} {'Min':<12} {'Max':<12} {'Media':<12} {'Desv.Std':<12}")
print(f"  {'-'*58}")

features = data_dict['features']

for i, feature in enumerate(features):
    col = X[:, i]
    print(f"  {feature:<10} {np.min(col):<12.4f} {np.max(col):<12.4f} "
          f"{np.mean(col):<12.4f} {np.std(col):<12.4f}")

# Balance de clases
clases, conteos = np.unique(YD, return_counts=True)
ratio_balance = min(conteos) / max(conteos)

print(f"\nBalance de clases:")
print(f"  Ratio min/max: {ratio_balance:.2f}")
if ratio_balance >= 0.8:
    print(f"  Estado: ✓ Bien balanceado")
elif ratio_balance >= 0.5:
    print(f"  Estado: ⚠ Moderadamente balanceado")
else:
    print(f"  Estado: ✗ Desbalanceado")

## 4. Partición del Dataset
Separar el dataset en tres conjuntos: entrenamiento (70%), validación (15%) y prueba (15%).

In [ ]:
n_total = len(X)
n_train = int(0.70 * n_total)
n_val   = int(0.15 * n_total)
n_test  = n_total - n_train - n_val

np.random.seed(42)
indices   = np.random.permutation(n_total)
train_idx = indices[:n_train]
val_idx   = indices[n_train:n_train+n_val]
test_idx  = indices[n_train+n_val:]

X_train  = X[train_idx];  YD_train = YD[train_idx]
X_val    = X[val_idx];    YD_val   = YD[val_idx]
X_test   = X[test_idx];   YD_test  = YD[test_idx]

print("=== SEPARACIÓN DEL DATASET ===")
print(f"Total de patrones: {n_total}")
print(f"  Train (70%):      {n_train}")
print(f"  Validation (15%): {n_val}")
print(f"  Test (15%):       {n_test}")
print(f"\nDimensiones:")
print(f"  X_train: {X_train.shape}, YD_train: {YD_train.shape}")
print(f"  X_val:   {X_val.shape},   YD_val:   {YD_val.shape}")
print(f"  X_test:  {X_test.shape},  YD_test:  {YD_test.shape}")

## 2. Parámetros de entradas y 5. Parámetros de entrenamiento
Configuración de la Red RBF

In [ ]:
# 2. Parámetros de entrada
n_entradas         = X_train.shape[1]
n_salidas          = YD_train.shape[1]
n_patrones         = X_train.shape[0]
# 5. Parámetros de entrenamiento
num_centros        = 20
error_optimo       = 0.05
max_iteraciones    = 10
incremento_centros = 3

print("=== CONFIGURACIÓN DE LA RED RBF ===")
print(f"Número de entradas:               {n_entradas}")
print(f"Número de salidas:                {n_salidas}")
print(f"Número de patrones de entren.:    {n_patrones}")

print(f"Número inicial de centros:        {num_centros}")
print(f"Función de activación:            Thin Plate Spline (FA = Ω² * ln(Ω))")
print(f"Error de aproximación óptimo:     {error_optimo}")
print(f"Máx. iteraciones reentrenamiento: {max_iteraciones}")
print(f"Incremento de centros/iteración:  {incremento_centros}")

## Funciones Auxiliares

In [ ]:
def inicializar_centros(X, num_centros):
    X_min = np.min(X, axis=0); X_max = np.max(X, axis=0)
    R = np.zeros((num_centros, X.shape[1]))
    for i in range(num_centros):
        for j in range(X.shape[1]):
            R[i, j] = np.random.uniform(X_min[j], X_max[j])
    return R

def calcular_distancias(X, R):
    D = np.zeros((X.shape[0], R.shape[0]))
    for p in range(X.shape[0]):
        for c in range(R.shape[0]):
            D[p, c] = np.sqrt(np.sum((X[p] - R[c]) ** 2))
    return D

def thin_plate_spline(omega):
    if omega <= 0: return 0
    return (omega ** 2) * np.log(omega)

def calcular_matriz_interpolacion(D):
    n_patrones, num_centros = D.shape
    FA = np.zeros((n_patrones, num_centros))
    for p in range(n_patrones):
        for c in range(num_centros):
            FA[p, c] = thin_plate_spline(D[p, c])
    A = np.zeros((n_patrones, num_centros + 1))
    A[:, 0] = 1;  A[:, 1:] = FA
    return A

def resolver_pesos(A, YD):
    return np.dot(np.linalg.pinv(A), YD)

def simular_red(X, R, W):
    return np.dot(calcular_matriz_interpolacion(calcular_distancias(X, R)), W)

def calcular_errores(YD, YR):
    EL  = YD - YR
    EG  = np.sum(np.abs(EL)) / len(YD)
    MSE = np.mean(EL ** 2)
    MSR = np.sum((YR - np.mean(YR)) ** 2) / len(YR)
    return EL, EG, MSE, MSR

def calcular_matriz_confusion(YD, YR):
    YD_c = np.round(YD.flatten()).astype(int)
    YR_c = np.round(YR.flatten()).astype(int)
    n_clases = max(np.max(YD_c), np.max(YR_c)) + 1
    mat = np.zeros((n_clases, n_clases), dtype=int)
    for i in range(len(YD_c)):
        mat[YD_c[i], YR_c[i]] += 1
    accuracy = np.sum(np.diag(mat)) / np.sum(mat) if np.sum(mat) > 0 else 0
    precisions, recalls, specificities, f1s = [], [], [], []
    for c in range(n_clases):
        TP = mat[c, c]
        FP = np.sum(mat[:, c]) - TP
        FN = np.sum(mat[c, :]) - TP
        TN = np.sum(mat) - TP - FP - FN
        prec = TP / (TP + FP) if (TP + FP) > 0 else 0
        rec  = TP / (TP + FN) if (TP + FN) > 0 else 0
        spec = TN / (TN + FP) if (TN + FP) > 0 else 0
        f1   = 2*(prec*rec)/(prec+rec) if (prec+rec) > 0 else 0
        precisions.append(prec); recalls.append(rec)
        specificities.append(spec); f1s.append(f1)
    return mat, accuracy, np.mean(precisions), np.mean(recalls), np.mean(specificities), np.mean(f1s)

def graficar_matriz_confusion(mat, class_names, title):
    sns.heatmap(mat, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicción'); plt.ylabel('Valor Real')
    plt.title(title); plt.tight_layout()

def imprimir_metricas_conjunto(nombre, n, acc, prec, rec, spec, f1, EG, MSE, MSR, mat):
    print(f"\n{nombre} ({n} patrones):")
    print(f"  Accuracy:    {acc:.4f}")
    print(f"  Precision:   {prec:.4f}")
    print(f"  Recall:      {rec:.4f}")
    print(f"  Specificity: {spec:.4f}")
    print(f"  F1-Score:    {f1:.4f}")
    print(f"  EG:          {EG:.6f}")
    print(f"  MSE:         {MSE:.6f}")
    print(f"  MSR:         {MSR:.6f}")
    print(f"  Matriz de Confusión:\n    {mat}")

def graficar_yd_yr(YD_set, YR_set, titulo):
    plt.figure(figsize=(10, 5))
    plt.plot(YD_set, label='Salida Deseada (YD)', marker='o', linestyle='None', alpha=0.6)
    plt.plot(YR_set, label='Salida Red (YR)',     marker='x', linestyle='None', alpha=0.6)
    plt.xlabel('Patrón'); plt.ylabel('Valor')
    plt.title(titulo); plt.legend(); plt.grid(True, alpha=0.3)
    plt.show()

def graficar_todo(tag, YD_train, YR_train, YD_val, YR_val, YD_test, YR_test,
                  mat_val, acc_val, f1_val, mat_test, acc_test, f1_test,
                  EG_obtenido, error_ref):
    """Genera todas las gráficas estándar para un entrenamiento (o reentrenamiento)."""

    # 1. EG vs Error Óptimo
    plt.figure(figsize=(8, 5))
    plt.bar(['EG Obtenido', 'Error Óptimo'], [EG_obtenido, error_ref],
            color=['steelblue', 'tomato'], alpha=0.8, edgecolor='black')
    plt.axhline(y=error_ref, color='red', linestyle='--', label=f'Error Óptimo ({error_ref})')
    plt.ylabel('Error')
    plt.title(f'[{tag}] EG Obtenido vs Error Óptimo')
    plt.legend(); plt.grid(True, alpha=0.3, axis='y')
    for i, v in enumerate([EG_obtenido, error_ref]):
        plt.text(i, v + 0.001, f'{v:.6f}', ha='center', va='bottom', fontsize=10)
    plt.tight_layout(); plt.show()

    # 2. YD vs YR para Train, Val, Test
    for YD_set, YR_set, nombre in [
        (YD_train, YR_train, 'Entrenamiento'),
        (YD_val,   YR_val,   'Validación'),
        (YD_test,  YR_test,  'Prueba')
    ]:
        graficar_yd_yr(YD_set, YR_set, f'[{tag}] Conjunto de {nombre}: YD vs YR')

    # 3. Matrices de confusión Val y Test
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    plt.sca(axes[0])
    graficar_matriz_confusion(mat_val,  ['Clase 0','Clase 1'],
                              f'[{tag}] Validation - Acc:{acc_val:.4f} F1:{f1_val:.4f}')
    plt.sca(axes[1])
    graficar_matriz_confusion(mat_test, ['Clase 0','Clase 1'],
                              f'[{tag}] Test - Acc:{acc_test:.4f} F1:{f1_test:.4f}')
    plt.tight_layout(); plt.show()

print("=== FUNCIONES AUXILIARES DEFINIDAS ===")

## 6. Entrenamiento de la Red RBF
RBF se resuelve directamente mediante pseudoinversa, sin ciclos iterativos.

In [ ]:
# Inicializar centros radiales
np.random.seed(42)
R = inicializar_centros(X_train, num_centros)

print("=== INICIALIZACIÓN DE CENTROS RADIALES ===")
print(f"Centros iniciales ({num_centros} centros):")
for i in range(num_centros):
    print(f"  R{i+1} = {R[i]}")

# Entrenar modelo
print("=== ENTRENAMIENTO DE LA RED RBF (UNA SOLA PASADA) ===\n")

np.random.seed(42)
# Se repite el entrenar centros
R = inicializar_centros(X_train, num_centros)
D = calcular_distancias(X_train, R)
A = calcular_matriz_interpolacion(D)
W = resolver_pesos(A, YD_train)

YR_train = simular_red(X_train, R, W)
EL_train, EG_train, MSE_train, MSR_train = calcular_errores(YD_train, YR_train)

# Guardar modelo
mejor_R  = R.copy()
mejor_W  = W.copy()
mejor_EG = EG_train
iteracion = 1

converge = EG_train <= error_optimo

print(f"Centros utilizados: {num_centros}")
print(f"\nResultados del entrenamiento:")
print(f"  Error General (EG): {EG_train:.6f}")
print(f"  Error Óptimo:       {error_optimo}")
print(f"  MSE:                {MSE_train:.6f}")
print(f"  MSR:                {MSR_train:.6f}")
print(f"\n{'✓ CONVERGIÓ' if converge else '✗ NO convergió'} con {num_centros} centros.")
if not converge:
    print("  → Ejecuta la sección 10 para reentrenar con más centros.")

## 7. Simular
Evaluación del Modelo en Train / Val / Test

In [ ]:
print("=== EVALUACIÓN DEL MODELO EN TRAIN / VAL / TEST ===")

YR_train = simular_red(X_train, mejor_R, mejor_W)
EL_train, EG_train, MSE_train, MSR_train = calcular_errores(YD_train, YR_train)

YR_val = simular_red(X_val, mejor_R, mejor_W)
EL_val, EG_val, MSE_val, MSR_val = calcular_errores(YD_val, YR_val)

YR_test = simular_red(X_test, mejor_R, mejor_W)
EL_test, EG_test, MSE_test, MSR_test = calcular_errores(YD_test, YR_test)

print(f"\n  {'Conjunto':<18} {'EG':<12} {'MSE':<12} {'MSR':<12}")
print(f"  {'-'*54}")
print(f"  {'Train (70%)':<18} {EG_train:<12.6f} {MSE_train:<12.6f} {MSR_train:<12.6f}")
print(f"  {'Validation (15%)':<18} {EG_val:<12.6f} {MSE_val:<12.6f} {MSR_val:<12.6f}")
print(f"  {'Test (15%)':<18} {EG_test:<12.6f} {MSE_test:<12.6f} {MSR_test:<12.6f}")

## 8. Matrices de Confusión y Métricas de Clasificación (Entrenamiento)

In [ ]:
print("=== MATRICES DE CONFUSIÓN Y 5 MÉTRICAS — ENTRENAMIENTO ===")

mat_val,  acc_val,  prec_val,  rec_val,  spec_val,  f1_val  = calcular_matriz_confusion(YD_val,  YR_val)
mat_test, acc_test, prec_test, rec_test, spec_test, f1_test = calcular_matriz_confusion(YD_test, YR_test)

imprimir_metricas_conjunto("VALIDATION SET", len(X_val),
    acc_val,  prec_val,  rec_val,  spec_val,  f1_val,
    EG_val,  MSE_val,  MSR_val,  mat_val)

imprimir_metricas_conjunto("TEST SET", len(X_test),
    acc_test, prec_test, rec_test, spec_test, f1_test,
    EG_test, MSE_test, MSR_test, mat_test)

## 9. Gráficas del Entrenamiento

In [ ]:
graficar_todo(
    tag='Entrenamiento',
    YD_train=YD_train, YR_train=YR_train,
    YD_val=YD_val,     YR_val=YR_val,
    YD_test=YD_test,   YR_test=YR_test,
    mat_val=mat_val,   acc_val=acc_val,   f1_val=f1_val,
    mat_test=mat_test, acc_test=acc_test, f1_test=f1_test,
    EG_obtenido=EG_train, error_ref=error_optimo
)